[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/04_rag_pipeline/04_rag_pipeline.ipynb)

# 04. RAG 파이프라인 실습 (임베딩 + 유사도 검색 + 프롬프트 조립)

> 관련 예제 프로젝트: [`example-projects/rag-regulation-example`](../../../example-projects/rag-regulation-example) (C파트: 규정 검색), `preprocess-example`의 인덱싱 부분

[01_web_crawling](../01_web_crawling/01_web_crawling.ipynb)에서 모으고 [02_text_chunking](../02_text_chunking/02_text_chunking.ipynb)에서 잘게 자른 문서 조각들을 검색해서 질문에 답하는 마지막 단계, RAG(검색 증강 생성)를 실습합니다.

## 이론

### 1) RAG란?
ChatGPT 같은 LLM은 회사 내부 규정 PDF를 학습한 적이 없어서 그 내용을 모릅니다. 그래서 관련 있는 문서 조각을 미리 찾아서(Retrieval) AI에게 "이거 보고 답해줘"라고 덧붙여 주는(Augmented) 방식으로 답을 생성(Generation)합니다 — 오픈북 시험과 비슷합니다.

### 2) 임베딩(embedding)
"연차휴가"와 "휴가 일수"는 사람이 보면 비슷한 말이지만, 컴퓨터는 글자만 봐서는 그걸 모릅니다. 문장을 숫자 좌표(벡터)로 바꿔서, 의미가 비슷한 문장일수록 좌표가 가깝게 나오도록 만드는 것이 임베딩입니다.

### 3) 벡터 검색과 OpenSearch
`rag-regulation-example`은 OpenSearch(`knn_vector` 필드)에 임베딩을 저장해두고, 질문 벡터와 가장 가까운 문서 조각을 찾습니다. 이 노트북은 Docker가 필요한 OpenSearch 없이도 실습할 수 있도록, **같은 개념을 numpy 코사인 유사도로 직접 구현**한 뒤, 실제 `opensearch-py`/OpenAI 연결 코드도 참고용으로 함께 보여줍니다 (연결 가능하면 실제로 사용, 아니면 자동으로 건너뜁니다).

## 실습 0. 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q numpy scikit-learn python-dotenv openai langchain-openai langchain-community opensearch-py

## 실습 1. 검색 대상 문서 조각 준비하기

[02_text_chunking.ipynb](../02_text_chunking/02_text_chunking.ipynb)에서 만든 것과 같은 사내 휴가 규정을 조항 단위로 미리 나눠 corpus로 사용합니다 (실제로는 `ingest.py`가 PDF를 청킹한 결과입니다).

In [ ]:
chunks = [
    {"source": "휴가규정.pdf", "page": 1,
     "text": "제2조(연차휴가) 1년간 80퍼센트 이상 출근한 임직원에게는 15일의 연차휴가를 부여한다. "
             "3년 이상 계속 근로한 임직원에 대하여는 매 2년마다 1일을 가산한 유급휴가를 부여한다."},
    {"source": "휴가규정.pdf", "page": 1,
     "text": "제3조(육아휴직) 만 8세 이하 자녀를 양육하는 임직원은 최대 1년의 육아휴직을 신청할 수 있다. "
             "육아휴직 기간은 근속연수에 포함하되, 승급을 위한 근속기간에는 산입하지 아니한다."},
    {"source": "휴가규정.pdf", "page": 2,
     "text": "제4조(경조휴가) 본인 결혼은 5일, 배우자 출산은 10일, 부모 사망은 5일의 유급휴가를 부여한다."},
    {"source": "휴가규정.pdf", "page": 2,
     "text": "제5조(병가) 업무 외 질병이나 부상으로 근로가 불가능한 경우 연간 60일 이내에서 병가를 "
             "사용할 수 있다. 병가 3일 이상 사용 시 진단서를 제출하여야 한다."},
    {"source": "복무규정.pdf", "page": 1,
     "text": "제10조(재택근무) 팀장의 승인을 받은 임직원은 주 2일까지 재택근무를 신청할 수 있다. "
             "재택근무 중 초과근무는 사전 승인 없이는 인정하지 않는다."},
]

print(f"{len(chunks)}개 문서 조각 준비 완료")

## 실습 2. 임베딩 함수 준비하기

`OPENAI_API_KEY`가 있으면 `config.py`의 `EMBEDDING_MODEL`(`text-embedding-3-small`)로 실제 임베딩을 만들고, 없으면 `scikit-learn`의 TF-IDF로 만든 벡터를 "의미 좌표"의 간이 대체재로 사용합니다. TF-IDF는 진짜 의미 기반 임베딩만큼 정교하지는 않지만, "텍스트 -> 숫자 벡터 -> 유사도 비교"라는 핵심 구조를 API 키 없이도 그대로 체험하게 해줍니다.

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv

load_dotenv()
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
USE_OPENAI = bool(os.getenv("OPENAI_API_KEY"))
print("실제 OpenAI 임베딩 사용:", USE_OPENAI)

In [ ]:
if USE_OPENAI:
    from openai import OpenAI
    _client = OpenAI()

    def embed_texts(texts: list[str]) -> np.ndarray:
        response = _client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return np.array([d.embedding for d in response.data])
else:
    from sklearn.feature_extraction.text import TfidfVectorizer

    _vectorizer = TfidfVectorizer()
    _vectorizer.fit([c["text"] for c in chunks])

    def embed_texts(texts: list[str]) -> np.ndarray:
        return _vectorizer.transform(texts).toarray()


chunk_vectors = embed_texts([c["text"] for c in chunks])
print("벡터 shape:", chunk_vectors.shape)

## 실습 3. 코사인 유사도로 유사 문서 검색하기

`query.py`의 `search_similar_docs()`와 같은 역할을 하는 함수를 직접 구현합니다. OpenSearch가 내부적으로 하는 "질문 벡터와 가장 가까운 문서 벡터 top-k 찾기"를 numpy로 그대로 재현합니다.

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_norm = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-8)
    return a_norm @ b_norm.T


def search_similar_docs(question: str, k: int = 4):
    query_vec = embed_texts([question])
    scores = cosine_similarity(query_vec, chunk_vectors)[0]
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top_k_idx]


results = search_similar_docs("육아휴직은 최대 몇 개월까지 쓸 수 있어?", k=3)
for doc, score in results:
    print(f"[{score:.3f}] {doc['source']} p.{doc['page']} - {doc['text'][:40]}...")

## 실습 4. `opensearch-py` 실제 연결 시도해보기 (선택)

Docker로 로컬 OpenSearch(`docker compose up -d`, `rag-regulation-example/docker-compose.yml` 참고)가 떠 있다면, 실제 `OpenSearchVectorSearch`(LangChain의 OpenSearch 벡터스토어 래퍼)로 위와 똑같은 검색을 할 수 있습니다. 이 노트북은 OpenSearch 없이도 끝까지 진행되도록, 연결에 실패하면 안내만 출력하고 넘어갑니다.

In [ ]:
OPENSEARCH_URL = os.getenv("OPENSEARCH_URL", "http://localhost:9200")

try:
    from opensearchpy import OpenSearch

    client = OpenSearch(OPENSEARCH_URL, timeout=2)
    info = client.info()
    print("OpenSearch 연결 성공:", info["version"]["number"])
    print("-> 이 경우 langchain_community.vectorstores.OpenSearchVectorSearch.from_documents(...)로")
    print("   실습 1~3과 동일한 인덱싱/검색을 실제 OpenSearch에서 수행할 수 있습니다 (ingest.py 참고).")
except Exception as e:
    print(f"OpenSearch에 연결할 수 없습니다 ({type(e).__name__}). "
          "로컬에 docker compose로 띄운 뒤 다시 실행하면 실제 연결을 확인할 수 있습니다.")
    print("지금은 실습 3의 numpy 코사인 유사도 검색으로 계속 진행합니다.")

## 실습 5. 프롬프트 조립하기 (그라운딩)

`query.py`의 `PROMPT_TEMPLATE`을 그대로 사용합니다. "아래 관련 규정만 근거로 답변하라"는 지시가 AI가 근거 없이 지어내는 것(환각, hallucination)을 막는 핵심 장치입니다.

In [ ]:
PROMPT_TEMPLATE = """당신은 사내 규정을 안내하는 어시스턴트입니다.
아래 [관련 규정]만 근거로 답변하고, 근거가 없으면 모른다고 답하세요.

[관련 규정]
{context}

[질문]
{question}

[답변]
"""


def build_prompt(question: str, docs) -> str:
    context = "\n\n".join(
        f"[{doc['source']} p.{doc['page']}]\n{doc['text']}" for doc, _score in docs
    )
    return PROMPT_TEMPLATE.format(context=context, question=question)


question = "육아휴직은 최대 몇 개월까지 쓸 수 있어?"
docs = search_similar_docs(question, k=3)
prompt = build_prompt(question, docs)
print(prompt)

## 실습 6. 답변 생성하기

`OPENAI_API_KEY`가 있으면 `query.py`처럼 `temperature=0`으로 실제 답변을 생성합니다. 없으면 조립된 프롬프트가 곧 "AI에게 넘어갈 최종 입력"이라는 점만 확인합니다.

In [ ]:
def answer(question: str, k: int = 4) -> str:
    docs = search_similar_docs(question, k=k)
    prompt = build_prompt(question, docs)

    if USE_OPENAI:
        from openai import OpenAI

        chat_client = OpenAI()
        chat_model = os.getenv("CHAT_MODEL", "gpt-4o-mini")
        response = chat_client.chat.completions.create(
            model=chat_model, temperature=0, messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

    return "(OPENAI_API_KEY가 없어 실제 답변 생성은 생략합니다. 위에서 조립된 프롬프트를 확인하세요.)"


print(answer("육아휴직은 최대 몇 개월까지 쓸 수 있어?"))

## 정리 & 연습 문제
- **연습 1**: `search_similar_docs()`의 `k`를 1, 3, 6으로 바꿔가며 어떤 문서가 새로 포함되거나 빠지는지, 유사도 점수가 어떻게 달라지는지 확인해보세요.
- **연습 2**: `chunks`에 전혀 없는 내용(예: "해외 출장 비용 정산 기준이 뭐야?")을 질문해서, 검색된 문서들의 유사도 점수가 낮게 나오는지 확인하고, `OPENAI_API_KEY`가 있다면 실제로 "모른다"는 답이 나오는지도 확인해보세요.

**해설/정답**: [04_rag_pipeline_solutions.ipynb](04_rag_pipeline_solutions.ipynb)

## 정리
이제 07~10 노트북으로 example-projects의 전체 파이프라인(크롤링 -> 청킹/인덱싱 -> 문서 구조화 -> RAG 검색/답변)에 쓰인 핵심 라이브러리를 모두 실습했습니다. 실제 프로젝트 코드(`example-projects/`)와 비교하며 읽어보면 이해가 더 깊어집니다.